# 01 — Explore Kaggle mdabbert Ultimate UFC Dataset

Sanity-check the processed Parquet before building features:
- Shape, date coverage, class balance
- Missingness per column group
- Odds sanity (no-vig implied probabilities, overround)
- Baseline: how well do the odds themselves predict the winner?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [ ]:
from ufc_pred.paths import PROCESSED

fights = pd.read_parquet(PROCESSED / 'fights.parquet')
upcoming = pd.read_parquet(PROCESSED / 'upcoming.parquet')

print(f'fights:   {fights.shape}')
print(f'upcoming: {upcoming.shape}')
print(f'date range: {fights.date.min().date()} → {fights.date.max().date()}')

## Class balance

In [ ]:
vc = fights['Winner'].value_counts()
print(vc)
print(f'\nRed-corner win rate: {(fights.Winner == "Red").mean():.3f}')
print('NOTE: red is the favored / higher-ranked fighter, so this is a matchmaking artifact, not a model signal.')

## Fights per year

In [ ]:
fights.groupby(fights.date.dt.year).size().rename('n_fights')

## Missingness — grouped by column family

118 columns. Group them by prefix to see which features are sparse.

In [ ]:
miss = fights.isna().mean().sort_values(ascending=False)
print('Top 20 most-missing columns:')
print(miss.head(20).round(3))

In [ ]:
groups = {
    'odds (moneyline)':     ['R_odds', 'B_odds'],
    'odds (method)':        ['r_dec_odds', 'b_dec_odds', 'r_sub_odds', 'b_sub_odds', 'r_ko_odds', 'b_ko_odds'],
    'fighter physical':     ['R_Height_cms', 'R_Reach_cms', 'R_Weight_lbs', 'B_Height_cms', 'B_Reach_cms', 'B_Weight_lbs'],
    'fighter career stats': ['R_avg_SIG_STR_landed', 'R_avg_TD_landed', 'B_avg_SIG_STR_landed', 'B_avg_TD_landed'],
    'rankings':             [c for c in fights.columns if c.endswith('_rank')],
    'finish details':       ['finish', 'finish_details', 'finish_round', 'finish_round_time', 'total_fight_time_secs'],
}

for name, cols in groups.items():
    cols = [c for c in cols if c in fights.columns]
    pct = fights[cols].isna().mean().mean()
    print(f'{name:25s}  avg missing: {pct:.1%}  ({len(cols)} cols)')

## Odds sanity check

`R_odds`/`B_odds` are American moneyline. Convert to decimal → implied probability → no-vig probability.

In [ ]:
def american_to_prob(odds):
    odds = pd.to_numeric(odds, errors='coerce')
    return np.where(odds > 0, 100 / (odds + 100), -odds / (-odds + 100))

df = fights.dropna(subset=['R_odds', 'B_odds', 'Winner']).copy()
df['p_R_raw'] = american_to_prob(df['R_odds'])
df['p_B_raw'] = american_to_prob(df['B_odds'])
df['overround'] = df['p_R_raw'] + df['p_B_raw']
df['p_R'] = df['p_R_raw'] / df['overround']
df['p_B'] = df['p_B_raw'] / df['overround']

print(f'Fights with odds:    {len(df):,} of {len(fights):,} ({len(df)/len(fights):.1%})')
print(f'Mean overround:      {df.overround.mean():.4f}  (1.0 = no vig; typical book ~1.04–1.08)')
print(f'Median overround:    {df.overround.median():.4f}')

## Baseline: how good are the odds themselves?

This is the bar Phase 1 must clear. If our model can't beat the implied probability of the odds, it has no business placing a bet.

In [ ]:
from sklearn.metrics import brier_score_loss, log_loss

scoreable = df[df.Winner.isin(['Red', 'Blue'])].copy()
y = (scoreable.Winner == 'Red').astype(int)
p = scoreable.p_R

acc = ((p > 0.5) == y).mean()
brier = brier_score_loss(y, p)
ll = log_loss(y, p.clip(1e-6, 1 - 1e-6))

print(f'Scored fights:   {len(scoreable):,}')
print(f'Pick favorite acc: {acc:.3f}')
print(f'Brier score:       {brier:.4f}  (lower better; 0.25 = coinflip)')
print(f'Log loss:          {ll:.4f}  (lower better; 0.693 = coinflip)')

## Calibration of the market

Bucket fights by implied probability of Red winning, check the empirical rate. If the market is calibrated, the bars should sit on the diagonal.

In [ ]:
buckets = pd.cut(scoreable.p_R, bins=np.arange(0, 1.05, 0.1))
cal = scoreable.groupby(buckets, observed=True).agg(
    n=('Winner', 'size'),
    predicted=('p_R', 'mean'),
    actual=('Winner', lambda s: (s == 'Red').mean()),
)
cal['diff'] = cal['actual'] - cal['predicted']
cal.round(3)

## Takeaways to record

Run the cells above, then jot down for the next session:
1. % of fights with odds (some early years probably missing)
2. Brier / log-loss of the market — this is **the** benchmark
3. Is the market well-calibrated, or systematically over/underpricing favorites?
4. Which feature groups are too sparse (>50% missing) to be useful in Phase 1?